In [3]:
# import libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import duckdb
from sklearn.model_selection import train_test_split
import pickle

#pio.templates.default = "seaborn"

pio.templates["custom_theme"] = go.layout.Template(
    layout=go.Layout(
        #paper_bgcolor='rgba(0,0,0,0)', 
        plot_bgcolor="rgba(230,230,230,255)",

        colorway=px.colors.qualitative.D3

    )
)

pio.templates.default = 'custom_theme'

In [4]:
with duckdb.connect("data/team_data.duckdb") as conn:
    df = conn.execute("SELECT * FROM ai4i2020").fetchdf()
    
df.head()    

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
# info about columns and not null values
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [6]:
# info about number of unique values => identification of potential categories
df.nunique()


UDI                        10000
Product ID                 10000
Type                           3
Air temperature [K]           93
Process temperature [K]       82
Rotational speed [rpm]       941
Torque [Nm]                  577
Tool wear [min]              246
Machine failure                2
TWF                            2
HDF                            2
PWF                            2
OSF                            2
RNF                            2
dtype: int64

In [7]:
df.columns.to_list()

['UDI',
 'Product ID',
 'Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'Machine failure',
 'TWF',
 'HDF',
 'PWF',
 'OSF',
 'RNF']

In [8]:
df_categories = [
    #'UDI',
    #'Product ID',
    'Type',
    #'Air temperature [K]',
    #'Process temperature [K]',
    #'Rotational speed [rpm]',
    #'Torque [Nm]',
    #'Tool wear [min]',
    'Machine failure',
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
    ]
for col in df_categories:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   UDI                      10000 non-null  int64   
 1   Product ID               10000 non-null  object  
 2   Type                     10000 non-null  category
 3   Air temperature [K]      10000 non-null  float64 
 4   Process temperature [K]  10000 non-null  float64 
 5   Rotational speed [rpm]   10000 non-null  int64   
 6   Torque [Nm]              10000 non-null  float64 
 7   Tool wear [min]          10000 non-null  int64   
 8   Machine failure          10000 non-null  category
 9   TWF                      10000 non-null  category
 10  HDF                      10000 non-null  category
 11  PWF                      10000 non-null  category
 12  OSF                      10000 non-null  category
 13  RNF                      10000 non-null  category
dtypes: cate

In [9]:
df.columns = df.columns.str.lower().str.replace(" ","_").str.replace("[","").str.replace("]","")
df.info()

lab_dict = {
    "udi":"UDI",
    "product_id":"Product ID",
    "type":"Type",
    "air_temperature_k":"Air temperature [K]",
    'process_temperature_k':'Process temperature [K]',
    'rotational_speed_rpm':'Rotational speed [RPM]',
    'torque_nm':'Torque [Nm]',
    'tool_wear_min':'Tool wear [min]',
    "machine_failure":"Machine failure",
    "twf":"TWF",
    "hdf":"HDF",
    "pwf":"PWF",
    "osf":"OSF",
    "rnf":"RNF"
    }

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   udi                    10000 non-null  int64   
 1   product_id             10000 non-null  object  
 2   type                   10000 non-null  category
 3   air_temperature_k      10000 non-null  float64 
 4   process_temperature_k  10000 non-null  float64 
 5   rotational_speed_rpm   10000 non-null  int64   
 6   torque_nm              10000 non-null  float64 
 7   tool_wear_min          10000 non-null  int64   
 8   machine_failure        10000 non-null  category
 9   twf                    10000 non-null  category
 10  hdf                    10000 non-null  category
 11  pwf                    10000 non-null  category
 12  osf                    10000 non-null  category
 13  rnf                    10000 non-null  category
dtypes: category(7), float64(3), int64(3), o

In [10]:
# Extract the labels
label_col = "machine_failure"
df_split = df.copy()
labels = np.array(df_split.pop(label_col))
RSEED = 42

# 30% examples in test data
train, test, train_labels, test_labels = train_test_split(df_split, labels, 
                                                          stratify = labels,
                                                          test_size = 0.3,
                                                          shuffle= True, 
                                                          random_state = RSEED)

In [11]:
train.reset_index(inplace = True)
df_train_labels = pd.DataFrame(train_labels)
df_train_labels.columns = ["machine_failure"]

df_train = pd.concat([train, df_train_labels ], axis=1)
df_train.machine_failure = df_train.machine_failure.astype("category")
df_train.drop("index", axis=1, inplace=True)
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   udi                    7000 non-null   int64   
 1   product_id             7000 non-null   object  
 2   type                   7000 non-null   category
 3   air_temperature_k      7000 non-null   float64 
 4   process_temperature_k  7000 non-null   float64 
 5   rotational_speed_rpm   7000 non-null   int64   
 6   torque_nm              7000 non-null   float64 
 7   tool_wear_min          7000 non-null   int64   
 8   twf                    7000 non-null   category
 9   hdf                    7000 non-null   category
 10  pwf                    7000 non-null   category
 11  osf                    7000 non-null   category
 12  rnf                    7000 non-null   category
 13  machine_failure        7000 non-null   category
dtypes: category(7), float64(3), int64(3), ob

In [12]:
df_train.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
udi,7000.0,NaN,NaN,NaN,4975.359143,2885.931018,1.0,2476.75,4962.5,7468.5,10000.0
product_id,7000,7000,M16748,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,7000,3,L,4217,NaN,NaN,NaN,NaN,NaN,NaN,NaN
air_temperature_k,7000.0,NaN,NaN,NaN,300.014357,2.002674,295.3,298.3,300.1,301.5,304.5
process_temperature_k,7000.0,NaN,NaN,NaN,310.006629,1.481637,305.7,308.8,310.1,311.1,313.8
rotational_speed_rpm,7000.0,NaN,NaN,NaN,1539.237429,180.237529,1168.0,1423.0,1504.0,1614.0,2886.0
torque_nm,7000.0,NaN,NaN,NaN,40.0013,10.006015,3.8,33.2,40.1,46.8,76.6
tool_wear_min,7000.0,NaN,NaN,NaN,107.623571,63.863906,0.0,52.0,107.0,163.0,253.0
twf,7000.0,2.0,0.0,6967.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hdf,7000.0,2.0,0.0,6927.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# matrix showing distribution of null values

# create binary matrix (1=present, 0=missing) from dataframe
matrix = df_train.notna().astype(int).values

# calculate missing percentages for column labels
missing_pct = df_train.isnull().sum() / len(df_train) * 100
x_labels = [f"{col}<br>({pct:.1f}% missing)" 
            for col, pct in zip(df_train.columns, missing_pct)]

# create chart
fig = go.Figure(data=go.Heatmap(
    z=matrix,
    x=x_labels,
    y=list(range(len(df_train))),
    colorscale=[[0, 'white'], [1, "black"]],
    showscale=False
))

fig.update_layout(
    #title='Missing Data Matrix',
    xaxis={'side': 'top', 'tickangle': -45},
    yaxis={'autorange': 'reversed'}
)

fig.show()

In [14]:
col_sort = "udi"
col_grouping = ["machine_failure", "type"]
size = 500
df_sorted = df_train.sort_values(col_sort)

for column in df_sorted.columns.to_list():
    for group in col_grouping:
        fig = px.histogram(
            data_frame = df_sorted,
            x = column,
            color = group,
            width = size,
            height = size,
            opacity=0.7,
            barmode="overlay",
            labels=lab_dict
            )
        fig.update_layout(legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.20,
        ))
        fig.show()

In [15]:
# pairplot for basic data distribution
fig_size = 2000

data_sort = "udi"
df_sorted = df_train.sort_values(data_sort)

pair_1 = px.scatter_matrix(
    df_sorted,
    dimensions=df.columns.to_list(),
    color='type',
    #symbol=,

    opacity=0.7,
    #size='value',
    #hover_data=['id', 'name']
    height=fig_size,
    width=fig_size,
    labels=lab_dict
)
pair_1.update_traces(showupperhalf=False)
pair_1.update_traces(diagonal_visible=True)
pair_1.update_traces(marker=dict(size=1))

pair_1.show()

In [16]:
df_train.columns.to_list()

['udi',
 'product_id',
 'type',
 'air_temperature_k',
 'process_temperature_k',
 'rotational_speed_rpm',
 'torque_nm',
 'tool_wear_min',
 'twf',
 'hdf',
 'pwf',
 'osf',
 'rnf',
 'machine_failure']

In [17]:
# pairplot for basic data distribution
fig_size = 1000

data_sort = "udi"

df_sorted = df_train.sort_values("udi")

par_2 = px.scatter_matrix(
    df_sorted,
    dimensions=[
        #'udi',
        #'product_id',
        #'type',
        'air_temperature_k',
        'process_temperature_k',
        'rotational_speed_rpm',
        'torque_nm',
        'tool_wear_min',
        'machine_failure',
        #'twf',
        #'hdf',
        #'pwf',
        #'osf',
        #'rnf'
        ],
    color='machine_failure',
    #symbol='group',  # Different symbols per group
    #labels=False,

    opacity=0.7,
    height=fig_size,
    width=fig_size,
    labels=lab_dict
)

# Hide upper triangle
par_2.update_traces(showupperhalf=False)

# Hide diagonal
par_2.update_traces(diagonal_visible=False)

# Customize marker size
par_2.update_traces(marker=dict(size=2))

par_2.show()

In [18]:
df_train["product_id"]

0       M16748
1       L52038
2       L56170
3       M19761
4       H37371
         ...  
6995    L48160
6996    L51446
6997    H37186
6998    L52960
6999    L48604
Name: product_id, Length: 7000, dtype: object

In [19]:
df_train["cons_number"] = df_train["product_id"].str.slice(start=1)
df_train["cons_number"] = df_train["cons_number"].astype("int") 

In [20]:
# individual scatter plots
size = 400
scatter_color ="machine_failure"
df_chart = df_train#[df_train["machine_failure"]==1]

scatter_pairs = {
    1:[ 'rotational_speed_rpm','torque_nm',  scatter_color],
    2:[ 'torque_nm', 'tool_wear_min', scatter_color],
    3:[ 'tool_wear_min','torque_nm',  scatter_color],
    4:['cons_number', 'udi',  scatter_color],
    5:[ 'udi','torque_nm',  scatter_color],
    6:[ 'cons_number','torque_nm',  scatter_color]

    }


for key, values in scatter_pairs.items():
    fig = px.scatter(
        df_chart,
        x = values[0],
        y = values[1],
        color = values[2],
        opacity = 0.7,
        width=size,
        height=size,
        labels=lab_dict

        #        color_continuous_scale = ["green", "red"]
    )
    fig.update_layout(legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.20,
   ))

    fig.show()


In [21]:

df_train_corr = df_train.select_dtypes(include=np.number).corr()


fig = px.imshow(
    df_train_corr, 
    text_auto=True, 
    color_continuous_scale=[px.colors.qualitative.D3[0],"white",px.colors.qualitative.D3[1]], 
    zmin=-1,  
    zmax=1,
    labels=lab_dict)

fig.show()


In [22]:
df_train.columns.to_list()

['udi',
 'product_id',
 'type',
 'air_temperature_k',
 'process_temperature_k',
 'rotational_speed_rpm',
 'torque_nm',
 'tool_wear_min',
 'twf',
 'hdf',
 'pwf',
 'osf',
 'rnf',
 'machine_failure',
 'cons_number']

In [23]:
size = 400

cols = [ 'air_temperature_k',
 'process_temperature_k',
 'rotational_speed_rpm',
 'torque_nm',
 'tool_wear_min',]
for col in cols:
    fig = px.box(
        df_train, 
        y=col, 
        width=size, 
        height=size, 
        labels=lab_dict)
    fig.show()

In [24]:
df.head()

,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [25]:
# Minimal feature engineering (no scaling, no encoding)
import numpy as np

# 1) Delta temperature: Process - Air
df["delta_temperature"] = df["process_temperature_k"] - df["air_temperature_k"]

# 2) Angular speed (rad/s) from rpm
df["omega"] = df["rotational_speed_rpm"] * (2 * np.pi / 60.0)

# 3) Calculated power (W) = torque * omega
df["calc_power"] = df["torque_nm"] * df["omega"]

# 4) Calculating power without calculating omega.
df["power"] = df["torque_nm"] * df["rotational_speed_rpm"]

In [26]:
df.head()

,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf,delta_temperature,omega,calc_power,power
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,162.420340,6951.590560,66382.8
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,147.445415,6826.722724,65190.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,156.870193,7749.387543,74001.2
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,150.063409,5927.504659,56603.5
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,147.445415,5897.816608,56320.0


In [27]:
df[["air_temperature_k", "process_temperature_k", "delta_temperature",  "rotational_speed_rpm", "omega", "torque_nm", "calc_power", "power" ]].head()

,air_temperature_k,process_temperature_k,delta_temperature,rotational_speed_rpm,omega,torque_nm,calc_power,power
0,298.1,308.6,10.5,1551,162.420340,42.8,6951.590560,66382.8
1,298.2,308.7,10.5,1408,147.445415,46.3,6826.722724,65190.4
2,298.1,308.5,10.4,1498,156.870193,49.4,7749.387543,74001.2
3,298.2,308.6,10.4,1433,150.063409,39.5,5927.504659,56603.5
4,298.2,308.7,10.5,1408,147.445415,40.0,5897.816608,56320.0


In [28]:
df_copy = df.copy()

In [29]:
# Features we will use for machine Failure
feature_cols = [
    "type",
    "air_temperature_k",
    "process_temperature_k",
    "delta_temperature",
    "rotational_speed_rpm",
    #"omega",
    "torque_nm",
    #"calc_power",
    "tool_wear_min",
    "power"
]

In [30]:
df["machine_failure"].describe()

count     10000
unique        2
top           0
freq       9661
Name: machine_failure, dtype: int64

In [31]:
from sklearn.model_selection import train_test_split
# X = inputs, y = target for Machine Failure
X = df[feature_cols]
y = df["machine_failure"]

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # stratify=y  It ensures the percentage of Machine_failure = 1 and Machine_failure = 0 is the same in both train and test sets.
)

In [32]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("positive in train:", int((y_train == 1).sum()))
print("positive in test :", int((y_test == 1).sum()))
#print("positive in train:", y_train.sum())
#print("positive in test :", y_test.sum())

Train shape: (7000, 8)
Test shape : (3000, 8)
positive in train: 237
positive in test : 102


In [33]:
numeric_columns = [
    "air_temperature_k",
    "process_temperature_k",
    "delta_temperature",
    "rotational_speed_rpm",
    #"omega",
    "torque_nm",
    #"calc_power",
    "tool_wear_min",
    "power"
]

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["type"]),
     ("num", "passthrough", numeric_columns)]
)

#### Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline_cmn = Pipeline(steps=[
   ("preprocess", preprocessor),
    ("random_forest", RandomForestClassifier(
        random_state=42,
        #n_jobs=-1
        n_estimators=400,
        #random_state=42,
        #class_weight= "balanced",
        n_jobs=-1
    ))
])

In [36]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

rf_pipeline_cmn.fit(X_train, y_train)

# Predict
y_pred_brf = rf_pipeline_cmn.predict(X_test)

In [37]:
# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred_brf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_brf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_brf))

Accuracy: 0.991

Confusion Matrix:
[[2894    4]
 [  23   79]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2898
           1       0.95      0.77      0.85       102

    accuracy                           0.99      3000
   macro avg       0.97      0.89      0.92      3000
weighted avg       0.99      0.99      0.99      3000



##### Randomforest With RandomizedSearch

In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline(steps=[
   ("preprocess", preprocessor),
    ("random_forest", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

In [39]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

param_dist = {
    "random_forest__class_weight": [ {0:1,1:5}, {0:1,1:10}, {0:1,1:20}, "balanced_subsample" ], # for Class 0 give importance 1 and for class1(Failures) importance 5
    "random_forest__min_samples_leaf": [1, 2, 5, 10],
    "random_forest__min_samples_split": [2, 5, 10, 20],
    "random_forest__max_depth": [None, 8, 12, 16],
    "random_forest__max_features": ["sqrt", "log2", 0.5, 0.7, 0.9],
    "random_forest__n_estimators": [400, 700, 1000],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    rf_pipeline, param_distributions=param_dist, n_iter=40,
    scoring="f1", cv=cv, n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train, y_train)
best = search.best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits


In [40]:
print(best)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['type']),
                                                 ('num', 'passthrough',
                                                  ['air_temperature_k',
                                                   'process_temperature_k',
                                                   'delta_temperature',
                                                   'rotational_speed_rpm',
                                                   'torque_nm', 'tool_wear_min',
                                                   'power'])])),
                ('random_forest',
                 RandomForestClassifier(class_weight={0: 1, 1: 5}, max_depth=16,
                                        max_features=0.9, min_samples_split=5,
                                        n_estimators=4

In [41]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred_best = best.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, digits=3))

Test Accuracy: 0.9916666666666667

Confusion Matrix:
[[2894    4]
 [  21   81]]

Classification Report:
              precision    recall  f1-score   support

           0      0.993     0.999     0.996      2898
           1      0.953     0.794     0.866       102

    accuracy                          0.992      3000
   macro avg      0.973     0.896     0.931      3000
weighted avg      0.991     0.992     0.991      3000



In [42]:
model_name = "random_forrest_optimized"
pickle.dump(best, open(f"models/{model_name}.sav","wb"))